<a href="https://colab.research.google.com/github/mutwassim/Detecting-New-and-Rare-Cyber-Attacks-on-NSL-KDD-/blob/main/NSL_KDD_Notebook1_Baselines.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# NSL-KDD Research — Notebook 1: Data check, reproduction, and honest baselines

This notebook covers **Weeks 1–4** of the research plan:

1. Download NSL-KDD and **verify row counts**.
2. **Experiment A — reproduce the target paper's protocol** (random 80:20 split of KDDTrain+, binary). You should see ~99% accuracy.
3. **Experiment B — official protocol** (train on KDDTrain+, test on KDDTest+ and KDDTest-21), binary and 5-class, 5 seeds. This is your honest baseline.
4. **Gate 1 check** against published official-split numbers.

**How to run:** `Runtime → Run all`. Everything runs on the free CPU runtime. Experiment B takes roughly 20–40 minutes; set `SEEDS = [0]` for a quick first run.

Target paper: Antony et al. (2026), *Discover Artificial Intelligence* 6:953 — https://link.springer.com/article/10.1007/s44163-026-01930-9

## 1. Setup

In [1]:
!pip -q install catboost xgboost lightgbm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 9.4 MB/s eta 0:00:00


In [2]:
import os, time, warnings, urllib.request
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder, RobustScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import (accuracy_score, f1_score, precision_score, recall_score,
                             confusion_matrix, classification_report)
warnings.filterwarnings('ignore')

SEEDS = [0, 1, 2, 3, 4]        # use [0] for a quick first run
RESULTS_DIR = 'results'
os.makedirs(RESULTS_DIR, exist_ok=True)

## 2. Download NSL-KDD and verify it

The official UNB page no longer hosts the files, so we use the GitHub mirror (jmnwong/NSL-KDD-Dataset).
If a download fails, download the three files manually from https://github.com/jmnwong/NSL-KDD-Dataset,
upload them to Colab (folder icon on the left), and re-run this cell.

In [3]:
BASE = 'https://raw.githubusercontent.com/jmnwong/NSL-KDD-Dataset/master/'
FILES = {'train': 'KDDTrain+.txt', 'test': 'KDDTest+.txt', 'test21': 'KDDTest-21.txt'}

for name in FILES.values():
    if not os.path.exists(name):
        url = BASE + name.replace('+', '%2B')
        try:
            urllib.request.urlretrieve(url, name)
            print('downloaded', name)
        except Exception as e:
            print('FAILED to download', name, '->', e, '\nUpload it manually and re-run.')

FEATURES = [
    'duration','protocol_type','service','flag','src_bytes','dst_bytes','land','wrong_fragment','urgent',
    'hot','num_failed_logins','logged_in','num_compromised','root_shell','su_attempted','num_root',
    'num_file_creations','num_shells','num_access_files','num_outbound_cmds','is_host_login','is_guest_login',
    'count','srv_count','serror_rate','srv_serror_rate','rerror_rate','srv_rerror_rate','same_srv_rate',
    'diff_srv_rate','srv_diff_host_rate','dst_host_count','dst_host_srv_count','dst_host_same_srv_rate',
    'dst_host_diff_srv_rate','dst_host_same_src_port_rate','dst_host_srv_diff_host_rate',
    'dst_host_serror_rate','dst_host_srv_serror_rate','dst_host_rerror_rate','dst_host_srv_rerror_rate']
COLS = FEATURES + ['attack', 'difficulty']
assert len(FEATURES) == 41

train = pd.read_csv(FILES['train'], names=COLS)
test = pd.read_csv(FILES['test'], names=COLS)
test21 = pd.read_csv(FILES['test21'], names=COLS)

EXPECTED = {'train': 125973, 'test': 22544, 'test21': 11850}
for name, df in [('train', train), ('test', test), ('test21', test21)]:
    status = 'OK' if len(df) == EXPECTED[name] else 'MISMATCH - check your file!'
    print(f'{name:7s} rows={len(df):7,d} cols={df.shape[1]}  expected={EXPECTED[name]:,d}  {status}')

downloaded KDDTrain+.txt
downloaded KDDTest+.txt
downloaded KDDTest-21.txt
train   rows=125,973 cols=43  expected=125,973  OK
test    rows= 22,544 cols=43  expected=22,544  OK
test21  rows= 11,850 cols=43  expected=11,850  OK


## 3. Labels: binary and 5 attack families

Each attack name is mapped to one of DoS, Probe, R2L, U2R (standard NSL-KDD mapping).
A few test-only attacks are mapped differently in some papers (e.g. `httptunnel`, `worm`);
**state your mapping in the paper**.

In [4]:
ATTACK_MAP = {
    'normal': 'normal',
    # DoS
    'back':'DoS','land':'DoS','neptune':'DoS','pod':'DoS','smurf':'DoS','teardrop':'DoS',
    'apache2':'DoS','mailbomb':'DoS','processtable':'DoS','udpstorm':'DoS',
    # Probe
    'ipsweep':'Probe','nmap':'Probe','portsweep':'Probe','satan':'Probe','mscan':'Probe','saint':'Probe',
    # R2L
    'ftp_write':'R2L','guess_passwd':'R2L','imap':'R2L','multihop':'R2L','phf':'R2L','spy':'R2L',
    'warezclient':'R2L','warezmaster':'R2L','sendmail':'R2L','named':'R2L','snmpgetattack':'R2L',
    'snmpguess':'R2L','xlock':'R2L','xsnoop':'R2L','httptunnel':'R2L','worm':'R2L',
    # U2R
    'buffer_overflow':'U2R','loadmodule':'U2R','perl':'U2R','rootkit':'U2R','ps':'U2R',
    'sqlattack':'U2R','xterm':'U2R',
}
CLASSES5 = ['normal', 'DoS', 'Probe', 'R2L', 'U2R']

for df in (train, test, test21):
    unknown = set(df['attack']) - set(ATTACK_MAP)
    assert not unknown, f'Unmapped attack names: {unknown}'
    df['family'] = df['attack'].map(ATTACK_MAP)
    df['binary'] = (df['family'] != 'normal').astype(int)

print('Class counts (5-class):')
print(pd.DataFrame({'KDDTrain+': train['family'].value_counts(),
                    'KDDTest+': test['family'].value_counts(),
                    'KDDTest-21': test21['family'].value_counts()}).reindex(CLASSES5))

test_only = sorted(set(test['attack']) - set(train['attack']))
n_test_only = test['attack'].isin(test_only).sum()
print(f'\nAttack types that appear ONLY in KDDTest+ ({len(test_only)}): {test_only}')
print(f'Rows with a test-only attack type: {n_test_only:,d} of {len(test):,d} ({n_test_only/len(test):.1%})')

Class counts (5-class):
        KDDTrain+  KDDTest+  KDDTest-21
family                                 
normal      67343      9711        2152
DoS         45927      7458        4342
Probe       11656      2421        2402
R2L           995      2887        2887
U2R            52        67          67

Attack types that appear ONLY in KDDTest+ (17): ['apache2', 'httptunnel', 'mailbomb', 'mscan', 'named', 'processtable', 'ps', 'saint', 'sendmail', 'snmpgetattack', 'snmpguess', 'sqlattack', 'udpstorm', 'worm', 'xlock', 'xsnoop', 'xterm']
Rows with a test-only attack type: 3,750 of 22,544 (16.6%)


## 4. Shared helpers

Preprocessing is always fitted on **training data only**, inside a Pipeline:
categorical columns → ordinal codes (unknown values → -1), numeric columns → RobustScaler.

In [5]:
CAT = ['protocol_type', 'service', 'flag']
NUM = [c for c in FEATURES if c not in CAT]

def make_preprocessor():
    return ColumnTransformer([
        ('cat', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), CAT),
        ('num', RobustScaler(), NUM),
    ])

def make_models(seed, n_classes):
    from xgboost import XGBClassifier
    from lightgbm import LGBMClassifier
    from catboost import CatBoostClassifier
    return {
        'RandomForest': RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=seed),
        'XGBoost': XGBClassifier(n_estimators=300, max_depth=8, learning_rate=0.1, tree_method='hist',
                                 n_jobs=-1, random_state=seed, verbosity=0),
        'LightGBM': LGBMClassifier(n_estimators=300, learning_rate=0.1, random_state=seed, verbose=-1),
        'CatBoost': CatBoostClassifier(iterations=500, random_seed=seed, verbose=0, thread_count=-1),
    }

def evaluate(y_true, y_pred, labels, positive_is_attack=True):
    out = {
        'accuracy': accuracy_score(y_true, y_pred),
        'macro_f1': f1_score(y_true, y_pred, average='macro', labels=labels, zero_division=0),
        'weighted_f1': f1_score(y_true, y_pred, average='weighted', labels=labels, zero_division=0),
    }
    rec = recall_score(y_true, y_pred, average=None, labels=labels, zero_division=0)
    for lab, r in zip(labels, rec):
        out[f'recall_{lab}'] = r
    # false-positive rate: normal traffic wrongly flagged as an attack
    normal = labels[0]
    is_normal = np.asarray(y_true) == normal
    out['FPR'] = float(np.mean(np.asarray(y_pred)[is_normal] != normal)) if is_normal.any() else np.nan
    if len(labels) == 2:
        out['attack_f1'] = f1_score(y_true, y_pred, pos_label=labels[1], zero_division=0)
    return out

## 5. Experiment A — reproduce the target paper's protocol (random 80:20 of KDDTrain+)

This mimics Antony et al. (2026): binary task, random stratified 80:20 split of the training file,
Random Forest and RBF SVM (C = 0.02, gamma = 0.04 as reported).

**Expected:** ~99% accuracy. Note that here the scaler is fitted on the training part only;
the paper appears to have scaled before splitting, which can only make its numbers slightly higher.

The SVM on ~100k rows is slow (can take 20+ minutes); `SVM_MAX_ROWS` trains it on a random subsample.
Set it to `None` to use everything.

In [6]:
SVM_MAX_ROWS = 20000
rows_A = []
X_all, y_all = train[FEATURES], train['binary']
for seed in SEEDS:
    Xtr, Xte, ytr, yte = train_test_split(X_all, y_all, test_size=0.2, stratify=y_all, random_state=seed)
    models = {'RandomForest': RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=seed),
              'SVM_RBF': SVC(C=0.02, gamma=0.04, kernel='rbf')}
    for name, model in models.items():
        Xfit, yfit = Xtr, ytr
        if name == 'SVM_RBF' and SVM_MAX_ROWS and len(Xtr) > SVM_MAX_ROWS:
            Xfit, _, yfit, _ = train_test_split(Xtr, ytr, train_size=SVM_MAX_ROWS, stratify=ytr, random_state=seed)
        pipe = Pipeline([('prep', make_preprocessor()), ('model', model)])
        t0 = time.time(); pipe.fit(Xfit, yfit); fit_s = time.time() - t0
        m = evaluate(yte, pipe.predict(Xte), labels=[0, 1])
        rows_A.append({'protocol': 'A: random 80:20', 'model': name, 'seed': seed, 'fit_seconds': fit_s, **m})
        print(f'seed {seed} {name:12s} acc={m["accuracy"]:.4f} attack_F1={m["attack_f1"]:.4f}')

res_A = pd.DataFrame(rows_A)
summary_A = res_A.groupby('model')[['accuracy', 'attack_f1', 'macro_f1', 'FPR']].agg(['mean', 'std'])
summary_A

seed 0 RandomForest acc=0.9988 attack_F1=0.9987
seed 0 SVM_RBF      acc=0.9617 attack_F1=0.9575
seed 1 RandomForest acc=0.9991 attack_F1=0.9991
seed 1 SVM_RBF      acc=0.9623 attack_F1=0.9583
seed 2 RandomForest acc=0.9988 attack_F1=0.9987
seed 2 SVM_RBF      acc=0.9665 attack_F1=0.9631
seed 3 RandomForest acc=0.9990 attack_F1=0.9989
seed 3 SVM_RBF      acc=0.9612 attack_F1=0.9569
seed 4 RandomForest acc=0.9990 attack_F1=0.9989
seed 4 SVM_RBF      acc=0.9567 attack_F1=0.9517


accuracy           attack_f1            macro_f1            \
                  mean       std      mean       std      mean       std   
model                                                                      
RandomForest  0.998936  0.000147  0.998857  0.000158  0.998931  0.000148   
SVM_RBF       0.961691  0.003503  0.957497  0.004040  0.961314  0.003563   

                   FPR            
                  mean       std  
model                             
RandomForest  0.000475  0.000113  
SVM_RBF       0.008479  0.001550

## 6. Experiment B — official protocol (train KDDTrain+ → test KDDTest+ and KDDTest-21)

Untuned default models first; tuning and class weighting come in Notebook 2.
Two tasks: **binary** (normal vs attack) and **5-class** (normal, DoS, Probe, R2L, U2R).

In [ ]:
rows_B = []
TESTS = {'KDDTest+': test, 'KDDTest-21': test21}
TASKS = {'binary': ('binary', [0, 1]), '5-class': ('family', CLASSES5)}

for task, (target, labels) in TASKS.items():
    ytr = train[target]
    if task == '5-class':
        enc = {c: i for i, c in enumerate(CLASSES5)}
        ytr = ytr.map(enc)
    for seed in SEEDS:
        for name, model in make_models(seed, len(labels)).items():
            pipe = Pipeline([('prep', make_preprocessor()), ('model', model)])
            t0 = time.time(); pipe.fit(train[FEATURES], ytr); fit_s = time.time() - t0
            for test_name, tdf in TESTS.items():
                pred = np.asarray(pipe.predict(tdf[FEATURES])).ravel()
                if task == '5-class':
                    pred = np.array(CLASSES5)[pred.astype(int)]
                m = evaluate(tdf[target].values, pred, labels=labels)
                rows_B.append({'protocol': 'B: official', 'task': task, 'test_set': test_name,
                               'model': name, 'seed': seed, 'fit_seconds': fit_s, **m})
            print(f'{task:8s} seed {seed} {name:12s} done ({fit_s:.0f}s)')

res_B = pd.DataFrame(rows_B)

binary   seed 0 RandomForest done (22s)
binary   seed 0 XGBoost      done (8s)
binary   seed 0 LightGBM     done (6s)
binary   seed 0 CatBoost     done (19s)
binary   seed 1 RandomForest done (21s)
binary   seed 1 XGBoost      done (9s)
binary   seed 1 LightGBM     done (5s)
binary   seed 1 CatBoost     done (18s)
binary   seed 2 RandomForest done (20s)
binary   seed 2 XGBoost      done (8s)
binary   seed 2 LightGBM     done (6s)
binary   seed 2 CatBoost     done (16s)
binary   seed 3 RandomForest done (21s)
binary   seed 3 XGBoost      done (8s)
binary   seed 3 LightGBM     done (6s)
binary   seed 3 CatBoost     done (17s)
binary   seed 4 RandomForest done (24s)
binary   seed 4 XGBoost      done (6s)
binary   seed 4 LightGBM     done (6s)
binary   seed 4 CatBoost     done (17s)
5-class  seed 0 RandomForest done (23s)
5-class  seed 0 XGBoost      done (33s)
5-class  seed 0 LightGBM     done (18s)
5-class  seed 0 CatBoost     done (66s)


### Results — binary

In [ ]:
cols_bin = ['accuracy', 'attack_f1', 'macro_f1', 'FPR', 'recall_1']
summary_B_bin = (res_B[res_B.task == 'binary']
                 .groupby(['test_set', 'model'])[cols_bin].agg(['mean', 'std']).round(4))
summary_B_bin

accuracy         attack_f1         macro_f1          \
                            mean     std      mean     std     mean     std   
test_set   model                                                              
KDDTest+   CatBoost       0.8047  0.0083    0.7983  0.0103   0.8045  0.0084   
           LightGBM       0.7878  0.0000    0.7771  0.0000   0.7873  0.0000   
           RandomForest   0.7757  0.0056    0.7610  0.0075   0.7749  0.0058   
           XGBoost        0.8034  0.0000    0.7968  0.0000   0.8032  0.0000   
KDDTest-21 CatBoost       0.6285  0.0157    0.7170  0.0154   0.5881  0.0131   
           LightGBM       0.5965  0.0000    0.6851  0.0000   0.5617  0.0000   
           RandomForest   0.5733  0.0107    0.6603  0.0115   0.5433  0.0087   
           XGBoost        0.6262  0.0000    0.7149  0.0000   0.5860  0.0000   

                            FPR         recall_1          
                           mean     std     mean     std  
test_set   model                                          
KDDTest+   CatBoost      0.0293  0.0005   0.6791  0.0143  
           LightGBM      0.0295  0.0000   0.6496  0.0000  
           RandomForest  0.0283  0.0003   0.6274  0.0100  
           XGBoost       0.0298  0.0000   0.6772  0.0000  
KDDTest-21 CatBoost      0.1321  0.0020   0.5754  0.0190  
           LightGBM      0.1329  0.0000   0.5364  0.0000  
           RandomForest  0.1275  0.0015   0.5069  0.0132  
           XGBoost       0.1334  0.0000   0.5728  0.0000

### Results — 5-class (look closely at R2L and U2R recall)

In [ ]:
cols_5 = ['accuracy', 'macro_f1', 'FPR'] + [f'recall_{c}' for c in CLASSES5]
summary_B_5 = (res_B[res_B.task == '5-class']
               .groupby(['test_set', 'model'])[cols_5].agg(['mean', 'std']).round(4))
summary_B_5

accuracy         macro_f1             FPR          \
                            mean     std     mean     std    mean     std   
test_set   model                                                            
KDDTest+   CatBoost       0.7652  0.0083   0.5514  0.0082  0.0279  0.0001   
           LightGBM       0.7391  0.0000   0.5064  0.0000  0.0695  0.0000   
           RandomForest   0.7473  0.0038   0.4925  0.0065  0.0271  0.0004   
           XGBoost        0.7704  0.0000   0.5455  0.0000  0.0297  0.0000   
KDDTest-21 CatBoost       0.5533  0.0157   0.4573  0.0104  0.1258  0.0006   
           LightGBM       0.5333  0.0000   0.4132  0.0000  0.1882  0.0000   
           RandomForest   0.5194  0.0071   0.3951  0.0075  0.1222  0.0019   
           XGBoost        0.5635  0.0000   0.4547  0.0000  0.1334  0.0000   

                        recall_normal         recall_DoS         recall_Probe  \
                                 mean     std       mean     std         mean   
test_set   model                                                                
KDDTest+   CatBoost            0.9721  0.0001     0.7998  0.0167       0.6834   
           LightGBM            0.9305  0.0000     0.7084  0.0000       0.7770   
           RandomForest        0.9729  0.0004     0.7808  0.0084       0.6378   
           XGBoost             0.9703  0.0000     0.8245  0.0000       0.6720   
KDDTest-21 CatBoost            0.8742  0.0006     0.6561  0.0287       0.6809   
           LightGBM            0.8118  0.0000     0.5180  0.0000       0.7752   
           RandomForest        0.8778  0.0019     0.6240  0.0143       0.6349   
           XGBoost             0.8666  0.0000     0.6988  0.0000       0.6694   

                                recall_R2L         recall_U2R          
                            std       mean     std       mean     std  
test_set   model                                                       
KDDTest+   CatBoost      0.0374     0.0632  0.0098     0.1254  0.0291  
           LightGBM      0.0000     0.1597  0.0000     0.0149  0.0000  
           RandomForest  0.0138     0.0102  0.0027     0.0388  0.0133  
           XGBoost       0.0000     0.0561  0.0000     0.1194  0.0000  
KDDTest-21 CatBoost      0.0377     0.0632  0.0098     0.1254  0.0291  
           LightGBM      0.0000     0.1597  0.0000     0.0149  0.0000  
           RandomForest  0.0139     0.0102  0.0027     0.0388  0.0133  
           XGBoost       0.0000     0.0561  0.0000     0.1194  0.0000

## 7. The headline comparison: random split vs official test set

In [ ]:
a = res_A[res_A.model == 'RandomForest'][['accuracy', 'attack_f1']].mean()
b = res_B[(res_B.task == 'binary') & (res_B.model == 'RandomForest')].groupby('test_set')[['accuracy', 'attack_f1']].mean()
comp = pd.DataFrame({'Random 80:20 (paper protocol)': a,
                     'Official KDDTest+': b.loc['KDDTest+'],
                     'Official KDDTest-21': b.loc['KDDTest-21']}).T.round(4)
print('Random Forest, binary task:')
comp

Random Forest, binary task:


,accuracy,attack_f1
Random 80:20 (paper protocol),0.9989,0.9989
Official KDDTest+,0.7757,0.7610
Official KDDTest-21,0.5733,0.6603


## 8. Gate 1 check

Published official-split references (binary):
- Gillani, Baig et al. 2026 (arXiv 2608.18155): attack-class F1 on KDDTest+ — Random Forest 0.764, best classical 0.803.

Your untuned baselines should land **roughly in the 0.70–0.85 range** for binary attack F1 on KDDTest+.
If you see ~0.99 on KDDTest+, something is wrong (e.g. test data leaked into training).

In [ ]:
rf_f1 = res_B[(res_B.task == 'binary') & (res_B.test_set == 'KDDTest+') & (res_B.model == 'RandomForest')]['attack_f1'].mean()
print(f'Your Random Forest attack-F1 on KDDTest+: {rf_f1:.3f}  (published reference: 0.764)')
if rf_f1 > 0.95:
    print('WARNING: suspiciously high. Check for leakage before continuing.')
elif 0.65 <= rf_f1 <= 0.88:
    print('GATE 1 PASSED: baselines are in the expected range.')
else:
    print('Outside the expected range. Re-check the files, label mapping and pipeline.')

Your Random Forest attack-F1 on KDDTest+: 0.761  (published reference: 0.764)
GATE 1 PASSED: baselines are in the expected range.


## 9. Where do the errors go? (confusion matrix, 5-class, KDDTest+)

In [ ]:
from xgboost import XGBClassifier
enc = {c: i for i, c in enumerate(CLASSES5)}
pipe = Pipeline([('prep', make_preprocessor()),
                 ('model', XGBClassifier(n_estimators=300, max_depth=8, learning_rate=0.1,
                                         tree_method='hist', n_jobs=-1, random_state=0, verbosity=0))])
pipe.fit(train[FEATURES], train['family'].map(enc))
pred = np.array(CLASSES5)[pipe.predict(test[FEATURES]).astype(int)]
cm = pd.DataFrame(confusion_matrix(test['family'], pred, labels=CLASSES5),
                  index=[f'true {c}' for c in CLASSES5], columns=[f'pred {c}' for c in CLASSES5])
print(classification_report(test['family'], pred, labels=CLASSES5, digits=3, zero_division=0))
cm

              precision    recall  f1-score   support

      normal      0.675     0.970     0.796      9711
         DoS      0.962     0.824     0.888      7458
       Probe      0.811     0.672     0.735      2421
         R2L      0.970     0.056     0.106      2887
         U2R      0.667     0.119     0.203        67

    accuracy                          0.770     22544
   macro avg      0.817     0.528     0.545     22544
weighted avg      0.822     0.770     0.730     22544



,pred normal,pred DoS,pred Probe,pred R2L,pred U2R
true normal,9423,81,205,1,1
true DoS,1156,6149,153,0,0
true Probe,631,163,1627,0,0
true R2L,2701,0,21,162,3
true U2R,55,0,0,4,8


### Recall on attack types never seen in training

In [ ]:
mask = test['attack'].isin(test_only)
unseen = pd.DataFrame({'attack': test.loc[mask, 'attack'], 'caught': pred[mask.values] != 'normal'})
print(f'Overall share of test-only attacks flagged as some attack: {unseen.caught.mean():.1%}')
unseen.groupby('attack')['caught'].agg(['count', 'mean']).sort_values('count', ascending=False).round(3)

Overall share of test-only attacks flagged as some attack: 34.9%


,count,mean
attack,,
mscan,996,0.368
apache2,737,0.634
processtable,685,0.172
snmpguess,331,0.000
saint,319,0.994
mailbomb,293,0.031
snmpgetattack,178,0.000
httptunnel,133,0.143
named,17,0.118


## 10. Save everything

Download the CSV files from the `results/` folder (left sidebar). Keep them — they become Table A and Table B in your paper.

In [ ]:
res_A.to_csv(f'{RESULTS_DIR}/experiment_A_random_split.csv', index=False)
res_B.to_csv(f'{RESULTS_DIR}/experiment_B_official_split.csv', index=False)
summary_B_bin.to_csv(f'{RESULTS_DIR}/summary_binary.csv')
summary_B_5.to_csv(f'{RESULTS_DIR}/summary_5class.csv')
cm.to_csv(f'{RESULTS_DIR}/confusion_5class_xgb_KDDTest+.csv')
print('Saved:', os.listdir(RESULTS_DIR))

Saved: ['experiment_B_official_split.csv', 'summary_5class.csv', 'summary_binary.csv', 'confusion_5class_xgb_KDDTest+.csv', 'experiment_A_random_split.csv']
